# try19res_py12 — Colab 学習ノートブック (F-1)

ローカルにGPUが無いので、学習だけ Colab(GPU) で回すためのノートブック。

**手順**
1. ランタイム → ランタイムのタイプを変更 → **GPU** を選択
2. 下の「GPU確認」を実行
3. コード取得（**A: GitHub clone** か **B: Drive のフォルダ** のどちらか）
4. 依存インストール
5. **スモーク学習**で短時間動作確認 → 問題なければ **本学習**
6. チェックポイントを Drive に保存（再接続しても消えないように）

> ローカルの VSCode で編集 → GitHub へ push → ここで clone、が一番楽です。

## 0. GPU確認

In [ ]:
!nvidia-smi

## 1. コード取得（A か B のどちらか一方を実行）

### A) GitHub から clone
事前にローカルの変更を push しておくこと（`try19res_py12` を含むコミット）。

In [ ]:
# === 方法A: GitHub clone ===
REPO_URL = "https://github.com/Waipy252/resnet-dqn.git"
BRANCH = "main"

%cd /content
![ -d resnet-dqn ] && rm -rf resnet-dqn
!git clone --branch $BRANCH --depth 1 $REPO_URL
%cd /content/resnet-dqn/try19res_py12
!pwd && ls

### B) Google Drive のフォルダを使う
`try19res_py12` フォルダを Drive の `MyDrive/resnet-dqn/` 等にアップしてある場合。
checkpoint もそのまま Drive に保存されるので再接続に強い。

In [ ]:
# === 方法B: Drive のフォルダを使う ===
from google.colab import drive
drive.mount('/content/drive')

# Drive 上のプロジェクトパスに合わせて書き換える
PROJECT_DIR = '/content/drive/MyDrive/resnet-dqn/try19res_py12'
%cd $PROJECT_DIR
!pwd && ls

## 2. 依存インストール
Colab には CUDA 版 torch がプリインストール済みなので **torch は触らない**（壊さない）。
旧 `gym` も入れない（gymnasium に移行済み）。

In [ ]:
!pip install -q \
    "stable-baselines3==2.8.0" \
    "sb3-contrib==2.8.0" \
    "gymnasium" "shimmy" \
    "yfinance" "pandas-datareader"
print('done')

In [ ]:
# import 確認 + デバイス確認
import torch, stable_baselines3, sb3_contrib, gymnasium
print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())
print('sb3', stable_baselines3.__version__, 'contrib', sb3_contrib.__version__)
import config
print('ALGO =', config.ALGO, '| REWARD_TYPE =', config.REWARD_TYPE, '| WINDOW =', config.WINDOW_SIZE)

## 3. スモーク学習（短時間で動作確認）
短い期間・少ステップで、データ取得→env→GPU学習→predict までを確認する。

In [ ]:
import torch
from stable_baselines3.common.vec_env import DummyVecEnv
import config
from data import generate_env_data
from main import NikkeiEnv, ResNetFeatures
from algo import build_model, get_algo_class

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device, '| algo =', get_algo_class().__name__)

# 短い期間でデータ取得（動作確認用）
df = generate_env_data('2015-01-01', '2024-01-01', ticker=config.TICKER)
print('data rows =', len(df))

env = DummyVecEnv([lambda: NikkeiEnv(df)])
model = build_model(env, device, features_extractor_class=ResNetFeatures, tensorboard_log=None)
model.learn(total_timesteps=3000, progress_bar=True)

obs = env.reset()
action, _ = model.predict(obs, deterministic=True)
print('smoke OK: device =', device, ' predicted action =', int(action))

## 4. 本学習
`main.py` をそのまま実行（config の設定で学習）。`config.TOTAL_TIMESTEPS`（既定35万）まで回り、
1万ステップごとに `nikkei_cp_..._<steps>_steps.zip` を保存する。

GPU メモリ/速度を見たいときは別タブのセルで `!nvidia-smi` を実行。

In [ ]:
!python main.py

## 5. チェックポイントを Drive に保存
（方法A で clone した場合、保存先はランタイム上なので消える。Drive にコピーして永続化する。）

In [ ]:
import glob, os, shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('drive mount skip:', e)

DST = '/content/drive/MyDrive/resnet-dqn-models'
os.makedirs(DST, exist_ok=True)
files = sorted(set(glob.glob('nikkei_cp_*.zip')))
for f in files:
    shutil.copy(f, DST)
print(f'copied {len(files)} files to {DST}')

## 6. （任意）評価
`eval.py` はテスト期間のデータを取得し、各チェックポイントをバックテストして指標を出す。
重いので動作確認だけなら飛ばしてよい。

In [ ]:
!python eval.py